In [1]:
import pandas as pd
df = pd.read_csv("IGCC_Uncertainty_dataset.csv")

In [2]:
df

,Ambient Air Temperature (Dry Bulb Average) (°F),As-Delivered Coal Cost ($/ton),CO2 Removal Efficiency (%),Power Requirement (% MWg),Cooling Water Temperature Drop (°F),Plant Outputs: Captured CO2: Flow Rate (tons/hr),Performance Parameter: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)
0,65.849998,58.299999,60.787498,8.295,28.900000,350.925293,566.327698,206.435303
1,80.550003,55.700001,85.112503,9.285,25.500000,463.870056,518.344238,150.798462
2,64.949997,62.900002,62.187500,6.435,47.299999,358.609619,578.199097,203.933243
3,37.950001,48.700001,86.512497,9.015,11.700000,496.849060,547.814941,144.885849
4,69.150002,31.500000,63.937500,6.495,37.500000,365.735321,572.414734,179.622055
...,...,...,...,...,...,...,...,...
195,32.549999,58.700001,69.362503,7.215,38.099998,411.700623,586.113220,179.689514
196,64.050003,51.099998,84.762497,9.555,41.500000,472.210388,528.255981,147.134338
197,43.650002,68.699997,65.512497,8.955,40.900002,385.882538,570.839600,196.145859
198,34.049999,49.900002,62.537498,11.565,17.900000,374.430664,562.361938,194.301773


In [3]:
df.isna().sum()

Ambient Air Temperature (Dry Bulb Average) (°F)                  0
As-Delivered Coal Cost ($/ton)                                   0
CO2 Removal Efficiency (%)                                       0
Power Requirement (% MWg)                                        0
Cooling Water Temperature Drop (°F)                              0
Plant Outputs: Captured CO2: Flow Rate (tons/hr)                 0
Performance Parameter: Net Electrical Output (MW)                0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Ambient Air Temperature (Dry Bulb Average) (°F)',
    'As-Delivered Coal Cost ($/ton)',
    'CO2 Removal Efficiency (%)',
    'Power Requirement (% MWg)',
    'Cooling Water Temperature Drop (°F)']]

y = df['Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)']

In [6]:
x

,Ambient Air Temperature (Dry Bulb Average) (°F),As-Delivered Coal Cost ($/ton),CO2 Removal Efficiency (%),Power Requirement (% MWg),Cooling Water Temperature Drop (°F)
0,65.849998,58.299999,60.787498,8.295,28.900000
1,80.550003,55.700001,85.112503,9.285,25.500000
2,64.949997,62.900002,62.187500,6.435,47.299999
3,37.950001,48.700001,86.512497,9.015,11.700000
4,69.150002,31.500000,63.937500,6.495,37.500000
...,...,...,...,...,...
195,32.549999,58.700001,69.362503,7.215,38.099998
196,64.050003,51.099998,84.762497,9.555,41.500000
197,43.650002,68.699997,65.512497,8.955,40.900002
198,34.049999,49.900002,62.537498,11.565,17.900000


In [7]:
y

0      206.435303
1      150.798462
2      203.933243
3      144.885849
4      179.622055
          ...    
195    179.689514
196    147.134338
197    196.145859
198    194.301773
199    158.211212
Name: Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== PYOMO MODEL ===\n")


=== PYOMO MODEL ===



In [9]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 4, 'reg__alpha': 0.001}
Train R2: 0.9999587319099364
Test R2: 0.9997442505054013


In [10]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Polynomial | RMSE: 0.3353 | MAPE: 0.1596%


In [11]:
import numpy as np, json, joblib

joblib.dump(best_poly, "IGCC_Cost_of_CO2_Captured_poly.pkl")

poly   = best_poly.named_steps['poly']
scaler = best_poly.named_steps['scaler']
reg    = best_poly.named_steps['reg']

coef_real      = reg.coef_ / scaler.scale_
intercept_real = reg.intercept_ - np.sum(reg.coef_ * scaler.mean_ / scaler.scale_)

feature_names = list(x.columns)
powers = poly.powers_
terms = [{"coef": float(c),
          "powers": {feature_names[j]: int(p[j]) for j in range(len(feature_names)) if p[j] > 0}}
         for p, c in zip(powers, coef_real)]
equation = {"intercept": float(intercept_real), "degree": int(poly.degree), "terms": terms}

P = poly.transform(x_test)
assert np.allclose(intercept_real + P @ coef_real, best_poly.predict(x_test)), "mismatch!"

json.dump(equation, open("IGCC_Cost_of_CO2_Captured_poly_eq.json", "w"), indent=2)
print("saved. degree =", equation["degree"], "| terms =", len(terms))

saved. degree = 4 | terms = 125
